# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)


**Lane:** Refresh / Content Opportunity Scoring

This contract defines the unit of analysis, fields, time windows, verification checks, and limits for a refresh-review ranking.

The decision point is the end of February 2026. February data is used for features and March data is used only for the outcome proxy.

## 0. Warehouse connection

The FlyRank warehouse is used for the verification queries below.

In [7]:
%pip -q install duckdb

import os
import duckdb
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = REL + "/fact_content_daily_performance"

FEB = "read_parquet('" + FACT + "/month=2026-02/*.parquet')"
MAR = "read_parquet('" + FACT + "/month=2026-03/*.parquet')"

print("Warehouse connection ready.")

Warehouse connection ready.


## 1. Unit of analysis + time window

One row in the final analysis represents one pseudonymized content item for one client at the February 2026 decision cutoff. February daily records are aggregated to the client × content level.

The feature window is **2026-02-01 through 2026-02-28**.

The outcome window is **2026-03-01 through 2026-03-31**.

The outcome proxy is `went_dark`: measured March GSC data with zero March GSC clicks. March performance is not used as a feature.

The output is a decision-support ranking for which content items to review first.

In [8]:
grain_check = con.sql(f"""
SELECT COUNT(*) AS duplicate_keys
FROM (
    SELECT client_hash_id, content_hash_id, report_date
    FROM {FEB}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) t
""").df()

feb_window = con.sql(f"""
SELECT COUNT(*) AS daily_rows,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM {FEB}
""").df()

mar_window = con.sql(f"""
SELECT COUNT(*) AS daily_rows,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM {MAR}
""").df()

print("February grain check:")
display(grain_check)
print("February window:")
display(feb_window)
print("March window:")
display(mar_window)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February grain check:


,duplicate_keys
0,0


February window:


,daily_rows,clients,content_items,first_date,last_date
0,7355108,54,321546,2026-02-01,2026-02-28


March window:


,daily_rows,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Features

- `impressions_feb` — February GSC impressions available by the decision cutoff.
- `clicks_feb` — February GSC clicks available by the decision cutoff.
- `ctr_feb` — February clicks divided by February impressions.
- `avg_position_feb` — February impression-weighted average position.
- `ga4_sessions_feb` — February GA4 sessions where GA4 data is available.

### Label / proxy

- `went_dark` — March outcome proxy equal to 1 when measured March GSC clicks are zero.

### Context

- `client_hash_id` — pseudonymized client identifier used for grouping and checking coverage.
- `content_hash_id` — pseudonymized content identifier used for grouping.
- `report_date` — source daily date used to verify the time window.

### Excluded

- March GSC/GA4 performance fields are excluded from features because they occur after the February decision cutoff and would leak future information.
- Client names, URLs, and other identifying fields are excluded from the analysis output.
- Raw daily records are not used directly as model rows; they are aggregated to the client × content decision unit.

In [9]:
schema_check = con.sql(f"DESCRIBE SELECT * FROM {FEB} LIMIT 1").df()

fields_used = [
    "client_hash_id", "content_hash_id", "report_date",
    "gsc_data_available", "gsc_impressions", "gsc_clicks",
    "gsc_sum_position", "ga4_data_available", "ga4_sessions"
]

display(
    schema_check[schema_check["column_name"].isin(fields_used)]
    [["column_name", "column_type"]]
)

feature_panel_check = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions ELSE 0 END) AS impressions_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS clicks_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_sum_position ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN gsc_data_available IS TRUE
                     THEN gsc_impressions ELSE 0 END), 0
        ) AS avg_position_feb,
        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS ga4_sessions_feb
    FROM {FEB}
    GROUP BY 1, 2
)
SELECT
    COUNT(*) AS decision_rows,
    COUNT(*) FILTER (WHERE impressions_feb > 0) AS rows_with_impressions,
    COUNT(*) FILTER (WHERE ga4_sessions_feb > 0) AS rows_with_ga4_sessions
FROM feb
""").df()

display(feature_panel_check)

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT
12,ga4_sessions,BIGINT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,decision_rows,rows_with_impressions,rows_with_ga4_sessions
0,321546,153559,33347


## 3. Verify it with queries

The checks below verify the grain, counts, missing/availability status, and feature/outcome windows.

In [10]:
grain_check = con.sql(f"""
SELECT COUNT(*) AS duplicate_keys
FROM (
    SELECT client_hash_id, content_hash_id, report_date
    FROM {FEB}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) t
""").df()

counts_check = con.sql(f"""
SELECT COUNT(*) AS feb_daily_rows,
       COUNT(DISTINCT client_hash_id) AS feb_clients,
       COUNT(DISTINCT content_hash_id) AS feb_content_items
FROM {FEB}
""").df()

availability_check = con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_measured_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable_or_null_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_measured_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_null_rows
FROM {FEB}
""").df()

window_check = con.sql(f"""
SELECT 'features_feb' AS window_name,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date,
       COUNT(*) AS rows
FROM {FEB}
UNION ALL
SELECT 'outcome_mar' AS window_name,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date,
       COUNT(*) AS rows
FROM {MAR}
ORDER BY window_name
""").df()

label_check = con.sql(f"""
WITH march AS (
    SELECT client_hash_id,
           content_hash_id,
           SUM(CASE WHEN gsc_data_available IS TRUE
                    THEN gsc_clicks ELSE 0 END) AS clicks_mar,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measured_gsc_days_mar
    FROM {MAR}
    GROUP BY 1, 2
)
SELECT COUNT(*) AS labeled_items,
       COUNT(*) FILTER (WHERE clicks_mar = 0) AS went_dark_items,
       COUNT(*) FILTER (WHERE clicks_mar > 0) AS non_dark_items,
       MIN(measured_gsc_days_mar) AS min_measured_gsc_days,
       MAX(measured_gsc_days_mar) AS max_measured_gsc_days
FROM march
WHERE measured_gsc_days_mar > 0
""").df()

print("Grain check:")
display(grain_check)
print("February counts:")
display(counts_check)
print("Availability checks:")
display(availability_check)
print("Feature and outcome windows:")
display(window_check)
print("March outcome proxy:")
display(label_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:


,duplicate_keys
0,0


February counts:


...

,total_rows,gsc_measured_rows,gsc_unavailable_or_null_rows,ga4_measured_rows,ga4_unavailable_or_null_rows
0,7355108,2621783,4733325,145321,7209787


Feature and outcome windows:


,window_name,first_date,last_date,rows
0,features_feb,2026-02-01,2026-02-28,7355108
1,outcome_mar,2026-03-01,2026-03-31,9841378


March outcome proxy:


,labeled_items,went_dark_items,non_dark_items,min_measured_gsc_days,max_measured_gsc_days
0,176738,107901,68837,1,31


## 4. Data limits

- Client history is not necessarily balanced across the warehouse, so the observed February panel does not imply equal historical coverage for every client.
- GSC and GA4 availability varies. An unavailable source is not treated as a measured zero.
- The March outcome proxy is only defined for content with measured March GSC data.
- The March proxy is a directional decision-support label; it does not establish why clicks were zero or whether a refresh would change the outcome.
- The February feature window and March outcome window are separated, but this contract covers one feature month and one outcome month rather than a long historical panel.
- Daily source records are aggregated to the client × content decision unit, so daily variation is not preserved in the final feature row.

In [11]:
dim_clients = REL + "/dim_clients.parquet"

history_check = con.sql(f"""
SELECT COUNT(*) AS clients,
       COUNT(DISTINCT gsc_data_start) AS distinct_gsc_start_dates,
       COUNT(DISTINCT ga4_data_start) AS distinct_ga4_start_dates,
       MIN(gsc_data_start) AS earliest_gsc_start,
       MAX(gsc_data_start) AS latest_gsc_start,
       MIN(ga4_data_start) AS earliest_ga4_start,
       MAX(ga4_data_start) AS latest_ga4_start
FROM read_parquet('{dim_clients}')
""").df()

overlap_check = con.sql(f"""
SELECT
    (SELECT MAX(report_date) FROM {FEB})
    < (SELECT MIN(report_date) FROM {MAR})
    AS windows_are_separate
""").df()

availability_limit_check = con.sql(f"""
SELECT
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND gsc_impressions IS NULL
    ) AS gsc_measured_but_null_impressions,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
          AND ga4_sessions IS NULL
    ) AS ga4_measured_but_null_sessions
FROM {FEB}
""").df()

print("Client history coverage:")
display(history_check)
print("Feature and outcome windows are separate:")
display(overlap_check)
print("Availability handling check:")
display(availability_limit_check)

Client history coverage:


,clients,distinct_gsc_start_dates,distinct_ga4_start_dates,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start
0,104,45,28,2025-01-27,2026-06-02,2025-10-29,2026-06-01


Feature and outcome windows are separate:


,windows_are_separate
0,True


Availability handling check:


,gsc_measured_but_null_impressions,ga4_measured_but_null_sessions
0,0,0


In [13]:
print("Grain check:")
display(grain_check)

print("February counts:")
display(counts_check)

print("Availability checks:")
display(availability_check)

print("Feature and outcome windows:")
display(window_check)

print("March outcome proxy:")
display(label_check)

display(feature_panel_check)

Grain check:


,duplicate_keys
0,0


February counts:


,feb_daily_rows,feb_clients,feb_content_items
0,7355108,54,321546


Availability checks:


...

Feature and outcome windows:


,window_name,first_date,last_date,rows
0,features_feb,2026-02-01,2026-02-28,7355108
1,outcome_mar,2026-03-01,2026-03-31,9841378


March outcome proxy:


,labeled_items,went_dark_items,non_dark_items,min_measured_gsc_days,max_measured_gsc_days
0,176738,107901,68837,1,31


,decision_rows,rows_with_impressions,rows_with_ga4_sessions
0,321546,153559,33347


## Self-check

- [x] Every section is filled with Markdown reasoning and supporting code.
- [x] No client names, URLs, or private queries are included in the analysis output.
- [x] Claims use careful language: observed, measured, proxy, and decision-support.
- [x] Future March outcome fields are excluded from February features.
- [x] Runtime → Run all completes without errors.
- [x] The executed notebook is committed under `work/notebooks/w03_data_contract.ipynb`.